# Sign Language Detection
### ASL Alphabet Classification

## 1. Install Dependencies

In [ ]:
!pip -q install tensorflow opencv-python matplotlib kaggle

## 2. Import Libraries

In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing import image_dataset_from_directory
from tensorflow.keras import layers, models

## 3. Download Dataset

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d grassknoted/asl-alphabet
!unzip -q asl-alphabet.zip -d asl_alphabet_dataset

## 4. Load Dataset

In [ ]:
dataset_path='./asl_alphabet_dataset/asl_alphabet_train/asl_alphabet_train'

train_ds=image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset='training',
    seed=42,
    image_size=(64,64),
    batch_size=32)

val_ds=image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset='validation',
    seed=42,
    image_size=(64,64),
    batch_size=32)

class_names=train_ds.class_names
print(class_names)
print(len(class_names))

## 5. Display Samples

In [ ]:
plt.figure(figsize=(8,8))
for images,labels in train_ds.take(1):
    images=images.numpy().astype('uint8')
    for i in range(9):
        plt.subplot(3,3,i+1)
        plt.imshow(images[i])
        plt.title(class_names[labels[i]])
        plt.axis('off')
plt.show()

## 6. Optimize Dataset

In [ ]:
AUTOTUNE=tf.data.AUTOTUNE
train_ds=train_ds.shuffle(1000).prefetch(AUTOTUNE)
val_ds=val_ds.prefetch(AUTOTUNE)

## 7. Build CNN

In [ ]:
model=models.Sequential([
layers.Input((64,64,3)),
layers.Rescaling(1./255),
layers.Conv2D(32,3,padding='same',activation='relu'),
layers.MaxPooling2D(),
layers.Conv2D(64,3,padding='same',activation='relu'),
layers.MaxPooling2D(),
layers.Conv2D(128,3,padding='same',activation='relu'),
layers.MaxPooling2D(),
layers.Flatten(),
layers.Dense(128,activation='relu'),
layers.Dropout(0.3),
layers.Dense(len(class_names))
])
model.summary()

## 8. Compile

In [ ]:
model.compile(
optimizer='adam',
loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
metrics=['accuracy'])

## 9. Train

In [ ]:
history=model.fit(train_ds,validation_data=val_ds,epochs=5)

## 10. Save

In [ ]:
model.save('sign_model.keras')